In [ ]:
import torch
import torch.nn as nn
import numpy as np

class AFF(nn.Module):

    def __init__(self, channels):
        super(AFF, self).__init__()

        self.local_att = nn.Sequential(
            nn.Linear(channels, channels),
            nn.ReLU(),
            nn.Linear(channels, channels)
        )

        self.global_att = nn.Sequential(
            nn.Linear(channels, channels),
            nn.ReLU(),
            nn.Linear(channels, channels)
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x, y):

        # 🔥 FORCE CORRECT SHAPE
        x = x.view(x.size(0), -1)
        y = y.view(y.size(0), -1)

        fusion = x + y

        local_weight = self.local_att(fusion)

        # ✔ CORRECT dim=1 usage
        global_context = fusion.mean(dim=1, keepdim=True)
        global_context = global_context.expand_as(fusion)

        global_weight = self.global_att(global_context)

        weight = self.sigmoid(local_weight + global_weight)

        out = weight * x + (1 - weight) * y

        return out

In [ ]:
train_swin = np.load("AC_X_train_swin.npy")
train_swin_Label = np.load("AC_y_train_swin.npy")

test_swin = np.load("AC_X_test_swin.npy")
test_swin_Label = np.load("AC_y_test_swin.npy")

train_resnet = np.load("AC_X_train_resnet.npy")
train_resnet_Label = np.load("AC_y_train_resnet.npy")

test_resnet = np.load("AC_X_test_resent.npy")
test_resnet_Label = np.load("AC_y_test_resnet.npy")
print(train_swin.shape)
print(train_swin_Label.shape)
print(test_swin.shape)
print(test_swin_Label.shape)
print(train_resnet.shape)
print(train_resnet_Label.shape)
print(test_resnet.shape)
print(test_resnet_Label.shape)

In [ ]:
print(train_resnet.shape)
print(train_swin.shape)
print(test_resnet.shape)
print(test_swin.shape)

In [ ]:
import torch

train_swin = torch.tensor(train_swin, dtype=torch.float32)
train_resnet = torch.tensor(train_resnet, dtype=torch.float32)

test_swin = torch.tensor(test_swin, dtype=torch.float32)
test_resnet = torch.tensor(test_resnet, dtype=torch.float32)



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
projection = nn.Linear(1024, 2048).to(device)
swin_train_proj = projection(train_swin)
swin_test_proj = projection(test_swin)

aff = AFF(2048)

fused_train_features = aff(train_resnet, swin_train_proj)
print(fused_train_features.shape)

testAff=AFF(2048)
fused_test_features = testAff(test_resnet, swin_test_proj)
print(fused_test_features.shape)

In [ ]:
# fused_train_features = fused_train_features.detach().cpu().numpy()

In [ ]:
# fused_test_features = fused_test_features.detach().cpu().numpy()

In [ ]:
np.save(
    "AC_AFF_Fused_Train_Features.npy",
    fused_train_features
)
np.save(
    "AC_AFF_Fused_Test_Features.npy",
    fused_test_features
)

print("AFF fusion completed successfully!")

In [ ]:
Train_labels = np.load("AC_y_train_resnet.npy")

np.save(
    "AC_AFF_Train_Labels.npy",
    Train_labels
)
Test_labels = np.load("AC_y_test_resnet.npy")

np.save(
    "AC_AFF_Test_Labels.npy",
    Test_labels
)